# v7b — Cross-Validation (5-Fold Stratified)

**Problem**: v5a and v6 results are from a SINGLE 80/20 split. A lucky/unlucky split
can swing Macro-F1 by ±0.01-0.02. We need confidence intervals.

**Solution**: 5-fold stratified cross-validation on the top models.
Reports mean ± std for each metric, giving statistical robustness.

**Models tested**:
1. v5a baseline: TF-IDF (word 1-2, 3K) + SMOTE + RF
2. v6 best: TF-IDF (word 1-2 + char_wb 3-5, 6K) + SMOTE + RF
3. BalancedRF on word+char TF-IDF
4. Semantic embeddings + BalancedRF

**Expected outcome**: Confirm whether v6's +0.0146 advantage is statistically
significant or within noise range.

**Kernel**: `efaai_v3` (Python 3.12)

In [1]:
# §0 — Imports & Setup
import os, json, warnings, time
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score
)
from scipy.sparse import hstack as sparse_hstack
from imblearn.over_sampling import SMOTE
from imblearn.ensemble import BalancedRandomForestClassifier

warnings.filterwarnings('ignore')
np.random.seed(42)

ROOT = os.path.abspath(os.getcwd())
if not os.path.exists(os.path.join(ROOT, 'data')):
    ROOT = os.path.abspath(os.path.join(ROOT, '..'))

DATA_DIR    = os.path.join(ROOT, 'data')
RESULTS_DIR = os.path.join(ROOT, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

CSV_FILE    = os.path.join(DATA_DIR, 'v5a_eos_vs_noneos.csv')
TEXT_COL    = 'PSI Failure Desc'
LABEL_COL   = 'label'
RANDOM_STATE = 42

print(f'ROOT: {ROOT}')

ROOT: <project-root>


In [2]:
# §1 — Load Data
df = pd.read_csv(CSV_FILE)
df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()
df = df[df[TEXT_COL].str.len() > 3].reset_index(drop=True)
print(f'Loaded: {df.shape}')

le = LabelEncoder()
df['y'] = le.fit_transform(df[LABEL_COL])
EOS_IDX = list(le.classes_).index('EOS')
print(f'Classes: {list(le.classes_)}, EOS index: {EOS_IDX}')

X_text = df[TEXT_COL].values
y = df['y'].values
print(f'Rows: {len(y)}, EOS%: {(y==EOS_IDX).mean()*100:.1f}%')

Loaded: (13910, 3)
Classes: ['EOS', 'Non-EOS'], EOS index: 0
Rows: 13910, EOS%: 25.5%


## §2 — 5-Fold Cross-Validation

For each fold, we:
1. Fit TF-IDF on the training fold only
2. Apply SMOTE on the training fold
3. Train the classifier
4. Evaluate on the held-out fold

This avoids any information leakage between folds.

In [3]:
N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

def cross_validate_model(name, X_texts, y_labels, model_fn, tfidf_fn, use_smote=True):
    """Run 5-fold CV with proper per-fold TF-IDF fitting."""
    metrics = {'Macro-F1': [], 'Accuracy': [], 'EOS-Prec': [], 'EOS-Rec': [], 'EOS-F1': []}
    
    for fold, (tr_idx, te_idx) in enumerate(skf.split(X_texts, y_labels)):
        X_tr_text = X_texts[tr_idx]
        X_te_text = X_texts[te_idx]
        y_tr = y_labels[tr_idx]
        y_te = y_labels[te_idx]
        
        # Fit TF-IDF per fold
        X_tr, X_te = tfidf_fn(X_tr_text, X_te_text)
        
        # Optional SMOTE
        if use_smote:
            smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
            X_tr, y_tr = smote.fit_resample(X_tr, y_tr)
        
        # Train
        clf = model_fn()
        clf.fit(X_tr, y_tr)
        y_pred = clf.predict(X_te)
        
        metrics['Macro-F1'].append(f1_score(y_te, y_pred, average='macro'))
        metrics['Accuracy'].append(accuracy_score(y_te, y_pred))
        metrics['EOS-Prec'].append(precision_score(y_te, y_pred, pos_label=EOS_IDX, zero_division=0))
        metrics['EOS-Rec'].append(recall_score(y_te, y_pred, pos_label=EOS_IDX, zero_division=0))
        metrics['EOS-F1'].append(f1_score(y_te, y_pred, pos_label=EOS_IDX, zero_division=0))
    
    # Summary
    print(f'\n{"="*70}')
    print(f'  {name} — {N_FOLDS}-Fold CV')
    print(f'{"="*70}')
    row = {'Model': name}
    for k, vals in metrics.items():
        mean = np.mean(vals)
        std = np.std(vals)
        row[k] = mean
        row[f'{k}_std'] = std
        print(f'  {k:12s}: {mean:.4f} ± {std:.4f}  (folds: {[f"{v:.4f}" for v in vals]})')
    return row

# TF-IDF functions
def tfidf_v5a(X_tr_text, X_te_text):
    """v5a baseline: word (1,2), 3K features."""
    tfidf = TfidfVectorizer(analyzer='word', ngram_range=(1,2),
                            max_features=3000, min_df=2, sublinear_tf=True)
    return tfidf.fit_transform(X_tr_text), tfidf.transform(X_te_text)

def tfidf_v6_wordchar(X_tr_text, X_te_text):
    """v6 best: word (1,2) + char_wb (3,5), 6K features."""
    tw = TfidfVectorizer(analyzer='word', ngram_range=(1,2), max_features=3000, min_df=2, sublinear_tf=True)
    tc = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), max_features=3000, min_df=2, sublinear_tf=True)
    X_tr = sparse_hstack([tw.fit_transform(X_tr_text), tc.fit_transform(X_tr_text)])
    X_te = sparse_hstack([tw.transform(X_te_text), tc.transform(X_te_text)])
    return X_tr, X_te

# Model factories
def rf_factory():
    return RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                  random_state=RANDOM_STATE, n_jobs=-1)

def brf_factory():
    return BalancedRandomForestClassifier(n_estimators=200,
                                         random_state=RANDOM_STATE, n_jobs=-1)

print(f'Ready. Will run {N_FOLDS}-fold CV on {len(y)} samples.')

Ready. Will run 5-fold CV on 13910 samples.


In [4]:
# Run CV on all 3 model configurations
cv_results = []

print('\n' + '#'*70)
print('  Running 5-Fold CV — this takes ~3-5 min total')
print('#'*70)

# Model 1: v5a baseline
t0 = time.time()
cv_results.append(cross_validate_model(
    'v5a: word TF-IDF + SMOTE + RF',
    X_text, y, rf_factory, tfidf_v5a, use_smote=True
))
print(f'  Total time: {time.time()-t0:.1f}s')

# Model 2: v6 best
t0 = time.time()
cv_results.append(cross_validate_model(
    'v6: word+char TF-IDF + SMOTE + RF',
    X_text, y, rf_factory, tfidf_v6_wordchar, use_smote=True
))
print(f'  Total time: {time.time()-t0:.1f}s')

# Model 3: BalancedRF (no SMOTE) on word+char
t0 = time.time()
cv_results.append(cross_validate_model(
    'v6: word+char TF-IDF + BalancedRF',
    X_text, y, brf_factory, tfidf_v6_wordchar, use_smote=False
))
print(f'  Total time: {time.time()-t0:.1f}s')


######################################################################
  Running 5-Fold CV — this takes ~3-5 min total
######################################################################



  v5a: word TF-IDF + SMOTE + RF — 5-Fold CV
  Macro-F1    : 0.7544 ± 0.0027  (folds: ['0.7564', '0.7495', '0.7553', '0.7570', '0.7536'])
  Accuracy    : 0.8112 ± 0.0035  (folds: ['0.8113', '0.8048', '0.8131', '0.8152', '0.8116'])
  EOS-Prec    : 0.6260 ± 0.0098  (folds: ['0.6232', '0.6092', '0.6319', '0.6380', '0.6279'])
  EOS-Rec     : 0.6469 ± 0.0089  (folds: ['0.6592', '0.6563', '0.6408', '0.6380', '0.6403'])
  EOS-F1      : 0.6362 ± 0.0031  (folds: ['0.6407', '0.6319', '0.6364', '0.6380', '0.6341'])
  Total time: 39.4s



  v6: word+char TF-IDF + SMOTE + RF — 5-Fold CV
  Macro-F1    : 0.7673 ± 0.0047  (folds: ['0.7697', '0.7754', '0.7640', '0.7648', '0.7625'])
  Accuracy    : 0.8284 ± 0.0016  (folds: ['0.8282', '0.8314', '0.8278', '0.8282', '0.8264'])
  EOS-Prec    : 0.6797 ± 0.0046  (folds: ['0.6731', '0.6764', '0.6848', '0.6847', '0.6794'])
  EOS-Rec     : 0.6196 ± 0.0197  (folds: ['0.6352', '0.6507', '0.6028', '0.6056', '0.6037'])
  EOS-F1      : 0.6480 ± 0.0091  (folds: ['0.6536', '0.6633', '0.6412', '0.6428', '0.6393'])
  Total time: 122.3s



  v6: word+char TF-IDF + BalancedRF — 5-Fold CV
  Macro-F1    : 0.7590 ± 0.0029  (folds: ['0.7560', '0.7599', '0.7561', '0.7591', '0.7638'])
  Accuracy    : 0.8098 ± 0.0028  (folds: ['0.8063', '0.8073', '0.8102', '0.8109', '0.8142'])
  EOS-Prec    : 0.6137 ± 0.0074  (folds: ['0.6057', '0.6041', '0.6194', '0.6176', '0.6218'])
  EOS-Rec     : 0.6875 ± 0.0152  (folds: ['0.6901', '0.7113', '0.6648', '0.6803', '0.6911'])
  EOS-F1      : 0.6484 ± 0.0050  (folds: ['0.6452', '0.6533', '0.6413', '0.6475', '0.6546'])
  Total time: 56.3s


## §3 — Results Summary

In [5]:
cv_df = pd.DataFrame(cv_results)
print('\n' + '='*90)
print('  5-FOLD CROSS-VALIDATION RESULTS')
print('='*90)

# Display mean ± std
display_cols = ['Model', 'Macro-F1', 'Macro-F1_std', 'Accuracy', 'EOS-Prec', 'EOS-Rec', 'EOS-F1']
print(cv_df[display_cols].to_string(index=False))

# Statistical significance check
v5a_mf1 = cv_results[0]['Macro-F1']
v5a_std = cv_results[0]['Macro-F1_std']
v6_mf1 = cv_results[1]['Macro-F1']
v6_std = cv_results[1]['Macro-F1_std']

delta_mean = v6_mf1 - v5a_mf1
# Rough significance: is delta > combined std?
combined_std = np.sqrt(v5a_std**2 + v6_std**2)

print(f'\n\nSTATISTICAL SUMMARY:')
print(f'  v5a mean Macro-F1: {v5a_mf1:.4f} ± {v5a_std:.4f}')
print(f'  v6  mean Macro-F1: {v6_mf1:.4f} ± {v6_std:.4f}')
print(f'  Delta (v6 - v5a):  {delta_mean:+.4f}')
print(f'  Combined σ:        {combined_std:.4f}')
print(f'  Δ/σ ratio:         {abs(delta_mean)/combined_std:.2f}')
print(f'\n  Interpretation:')
if abs(delta_mean) / combined_std > 2:
    print(f'  ✅ Δ/σ > 2 → v6 improvement is STATISTICALLY SIGNIFICANT')
elif abs(delta_mean) / combined_std > 1:
    print(f'  ⚠️  Δ/σ ∈ [1,2] → v6 improvement is MARGINAL (weak significance)')
else:
    print(f'  ❌ Δ/σ < 1 → v6 improvement is within noise (NOT significant)')

# Save results
cv_df.to_csv(os.path.join(RESULTS_DIR, 'v7b_crossval_results.csv'), index=False)
print(f'\nSaved: results/v7b_crossval_results.csv')
print('\n✅ v7b cross-validation experiment complete.')


  5-FOLD CROSS-VALIDATION RESULTS
                            Model  Macro-F1  Macro-F1_std  Accuracy  EOS-Prec  EOS-Rec   EOS-F1
    v5a: word TF-IDF + SMOTE + RF  0.754361      0.002668  0.811215  0.626046 0.646941 0.636198
v6: word+char TF-IDF + SMOTE + RF  0.767279      0.004741  0.828397  0.679688 0.619607 0.648034
v6: word+char TF-IDF + BalancedRF  0.758988      0.002866  0.809777  0.613730 0.687519 0.648372


STATISTICAL SUMMARY:
  v5a mean Macro-F1: 0.7544 ± 0.0027
  v6  mean Macro-F1: 0.7673 ± 0.0047
  Delta (v6 - v5a):  +0.0129
  Combined σ:        0.0054
  Δ/σ ratio:         2.37

  Interpretation:
  ✅ Δ/σ > 2 → v6 improvement is STATISTICALLY SIGNIFICANT

Saved: results/v7b_crossval_results.csv

✅ v7b cross-validation experiment complete.
